In [ ]:
# Bài 24 — RNN context vs Attention context
# So sánh LSTM final hidden state với self-attention context pooling trên IMDb.

import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model

# Keras tải IMDb đã token hóa từ Google Cloud Storage, không cần kết nối máy chủ Stanford.
MAX_VOCAB_SIZE = 20_000
MAX_LEN = 200
EMBEDDING_DIM = 64
CONTEXT_DIM = 64
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001
SEED = 42

(all_train_sequences, all_train_labels), (test_sequences, y_test) = (
    tf.keras.datasets.imdb.load_data(num_words=MAX_VOCAB_SIZE, seed=SEED)
)
all_train_labels = np.asarray(all_train_labels, dtype=np.int32)
y_test = np.asarray(y_test, dtype=np.int32)

# Tách validation theo chỉ số để giữ các sequence có độ dài khác nhau.
train_indices, val_indices = train_test_split(
    np.arange(len(all_train_sequences)), test_size=0.1, random_state=SEED,
    stratify=all_train_labels
)
train_sequences = [all_train_sequences[i] for i in train_indices]
val_sequences = [all_train_sequences[i] for i in val_indices]
y_train = all_train_labels[train_indices]
y_val = all_train_labels[val_indices]
vocab_size = MAX_VOCAB_SIZE

def encode_sequences(sequences):
    return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = encode_sequences(train_sequences)
X_val = encode_sequences(val_sequences)
X_test = encode_sequences(test_sequences)
print('Vocabulary size:', vocab_size)
print('Train:', X_train.shape, '| Validation:', X_val.shape, '| Test:', X_test.shape)
print('Train label counts (neg=0, pos=1):', np.bincount(y_train))

def build_lstm_model():
    tokens = layers.Input(shape=(MAX_LEN,), dtype='int32', name='tokens')
    mask = layers.Lambda(lambda x: tf.not_equal(x, 0), output_shape=(MAX_LEN,), name='valid_token_mask')(tokens)
    embedded = layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True, name='embedding')(tokens)
    context = layers.LSTM(CONTEXT_DIM, name='lstm_final_state')(embedded, mask=mask)
    output = layers.Dense(1, activation='sigmoid', name='sentiment')(context)
    model = Model(tokens, output, name='LSTM_final_context')
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_attention_model():
    tokens = layers.Input(shape=(MAX_LEN,), dtype='int32', name='tokens')
    valid = layers.Lambda(lambda x: tf.not_equal(x, 0), output_shape=(MAX_LEN,), name='valid_token_mask')(tokens)
    embedded = layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=False, name='embedding')(tokens)
    # Padding không được dùng làm key/value trong self-attention.
    key_mask = layers.Lambda(
        lambda m: tf.tile(tf.expand_dims(m, axis=1), [1, MAX_LEN, 1]),
        output_shape=(MAX_LEN, MAX_LEN), name='self_attention_mask'
    )(valid)
    attended = layers.MultiHeadAttention(
        num_heads=1, key_dim=CONTEXT_DIM, dropout=0.0, name='self_attention'
    )(query=embedded, value=embedded, key=embedded, attention_mask=key_mask)
    contextual_tokens = layers.LayerNormalization(name='attention_norm')(embedded + attended)
    # Học trọng số token rồi tổng hợp các vector thành context vector.
    scores = layers.Dense(1, name='pooling_score')(contextual_tokens)
    scores = layers.Lambda(
        lambda args: tf.where(args[1][..., None], args[0], tf.cast(-1e9, args[0].dtype)),
        output_shape=(MAX_LEN, 1), name='mask_pooling_scores'
    )([scores, valid])
    weights = layers.Softmax(axis=1, name='context_attention_weights')(scores)
    context = layers.Lambda(
        lambda args: tf.reduce_sum(args[0] * args[1], axis=1),
        output_shape=(CONTEXT_DIM,), name='attention_context'
    )([contextual_tokens, weights])
    context = layers.Dense(CONTEXT_DIM, activation='tanh', name='context_projection')(context)
    output = layers.Dense(1, activation='sigmoid', name='sentiment')(context)
    model = Model(tokens, output, name='SelfAttention_context')
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

models_to_compare = {
    'LSTM final hidden state': build_lstm_model,
    'Self-attention context pooling': build_attention_model,
}
results = []
trained_models = {}
for name, factory in models_to_compare.items():
    print(f'\n===== {name} =====')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = factory()
    parameter_count = model.count_params()
    start_time = time.perf_counter()
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True, verbose=2)
    training_seconds = time.perf_counter() - start_time
    probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(np.int32)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'Training time (seconds)': training_seconds,
        'Parameters': parameter_count,
    })
    trained_models[name] = model

results_df = pd.DataFrame(results)
print('\nBảng so sánh trên test set:')
display(results_df.round(4))
OUTPUT_CSV = '/content/lab06_bai24_results.csv'
results_df.to_csv(OUTPUT_CSV, index=False)
print('Đã lưu bảng kết quả:', OUTPUT_CSV)

# Câu hỏi cuối bài: Representation của hai model khác nhau ở đâu?
print('''
LSTM đọc chuỗi tuần tự; final hidden state là vector cố định được cập nhật qua từng bước và tổng hợp thông tin đã đi qua chuỗi. Self-attention tạo biểu diễn contextualized cho từng token bằng cách kết hợp thông tin từ các token khác; attention pooling tổng hợp các vector token thành context vector cho classifier. Vì vậy LSTM có một trạng thái cuối được cập nhật tuần tự, còn attention giữ biểu diễn theo vị trí trước bước tổng hợp và học trọng số để kết hợp chúng.
''')

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Vocabulary size: 20000
Train: (22500, 200) | Validation: (2500, 200) | Test: (25000, 200)
Train label counts (neg=0, pos=1): [11250 11250]

===== LSTM final hidden state =====
Epoch 1/3
352/352 - 39s - 111ms/step - accuracy: 0.7858 - loss: 0.4468 - val_accuracy: 0.8384 - val_loss: 0.3746
Epoch 2/3
352/352 - 37s - 104ms/step - accuracy: 0.9000 - loss: 0.2667 - val_accuracy: 0.8452 - val_loss: 0.3761
Epoch 3/3
352/352 - 37s - 104ms/step - accuracy: 0.9229 - loss: 0.2086 - val_accuracy: 0.8328 - val_loss: 0.4203

===== Self-attention context pooling =====
Epoch 1/3
352/352 - 44s - 125ms/step - accuracy: 0.8395 - loss: 0.3612 - val_accuracy: 0.8796 - val_loss: 0.2931
Epoch 2/3
352/352 - 44s - 124ms/step - accuracy: 0.9340 - loss: 0.1787 - val_accuracy: 0.8648 - val_loss: 0.3556
Epoch 3/3
352/352 - 82s - 232ms/step - accuracy: 0.9645 - loss: 0.1000 - val_accuracy: 0.8440 - val_loss: 0.4782

Bảng so sánh trên test set:


,Model,Accuracy,F1,Training time (seconds),Parameters
0,LSTM final hidden state,0.7983,0.7656,116.5986,1313089
1,Self-attention context pooling,0.7957,0.7649,169.2912,1301058


Đã lưu bảng kết quả: /content/lab06_bai24_results.csv

LSTM đọc chuỗi tuần tự; final hidden state là vector cố định được cập nhật qua từng bước và tổng hợp thông tin đã đi qua chuỗi. Self-attention tạo biểu diễn contextualized cho từng token bằng cách kết hợp thông tin từ các token khác; attention pooling tổng hợp các vector token thành context vector cho classifier. Vì vậy LSTM có một trạng thái cuối được cập nhật tuần tự, còn attention giữ biểu diễn theo vị trí trước bước tổng hợp và học trọng số để kết hợp chúng.

